In [1]:
import polars as pl
import pandas as pd

In [2]:
SCENARIO_NAMES_DICT = {
    1: 'Midden',
    2: 'VT',
    3: 'Elektrificatie',
    4: 'Waterstof',
    5: 'Groen gas'
}

REFERENCE_YEAR = '2024'

In [3]:
scenario = SCENARIO_NAMES_DICT[1]
print(scenario)

Midden


In [4]:
test_file_path = '/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/Shell Chemie Moerdijk.xlsx'

In [5]:
x = pl.read_excel('/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/Shell Chemie Moerdijk.xlsx',
              sheet_name=f'Scenario {scenario}')

In [45]:
import polars as pl
from openpyxl import load_workbook



def read_scenario_sheet(
    workbook_path: str,
    sheet_name: str,
    emission_cols: list[str],
    energy_cols: list[str],
    reference_year: int,
) -> pl.DataFrame:
    """
    Read a Scenario X sheet created by write_scenario_sheets().

    Returns a normalized dataframe with:
        Strategy
        Year
        Flow type
        <emission cols>
        <energy cols>

    - Skips the reference year
    - Handles merged year cells
    - Keeps rows even if all values are blank
    - Ignores separator rows
    """

    wb = load_workbook(workbook_path, data_only=True)
    ws = wb[sheet_name]

    scenario = sheet_name.replace("Scenario ", "")

    records = []
    current_year = None

    emission_start = 4
    energy_start = emission_start + len(emission_cols)

    for row_idx in range(5, ws.max_row + 1):

        # Year column (merged vertically)
        year_cell = ws.cell(row_idx, 2).value
        if year_cell is not None:
            current_year = str(year_cell)

        # Flow type column
        flow_type = ws.cell(row_idx, 3).value

        # Skip separator rows
        if flow_type is None:
            continue

        # Skip reference year
        if current_year == str(reference_year):
            continue

        record = {
            "Scenario": scenario,
            "Year": current_year,
            "Flow type": str(flow_type).lower(),
        }

        # Emissions
        for i, col in enumerate(emission_cols):
            record[col] = ws.cell(
                row_idx,
                emission_start + i
            ).value

        # Energy
        for i, col in enumerate(energy_cols):
            record[col] = ws.cell(
                row_idx,
                energy_start + i
            ).value

        # Keep row even if all values are empty
        records.append(record)

    if not records:
        return pl.DataFrame(
            schema={
                "Scenario": pl.Utf8,
                "Year": pl.Utf8,
                "Flow type": pl.Utf8,
                **{c: pl.Float64 for c in emission_cols},
                **{c: pl.Float64 for c in energy_cols},
            }
        )

    return pl.DataFrame(records)

In [46]:
STRICT_EMISSIONS_ORDER = ["CO2", "Methane", "N2O", "F-gases"]
EMISSION_COLS_ORDER =  STRICT_EMISSIONS_ORDER + ["CO2 (fossil) CCU/CCS", "CO2 (bio) CCU/CCS"]#, "other"]

# Single source of truth — used in both energy balance and project sheets
UTILITY_COLS_ORDER = [
    'Electricity',
    'Electricity_peak',
    'Natural Gas',
    'Hydrogen ( >98% vol.%) (LHV)',
    'Hydrogen ( <98% vol.%) (LHV)',
    'Heat',
    'Residual gases',
    'Coal and coal products',
    'Oil and oil products',
    'Biomass (liquid)',
    'Biomass (solid)',
    'Green gas',
    'Waste (fossil)',
    'Waste (bio)',
    'Ammonia',
    'Methanol',
    'Other syn fuel and raw materials',
    'Other',
]

In [48]:
df = read_scenario_sheet(
    test_file_path,
    f'Scenario {scenario}',
    EMISSION_COLS_ORDER,
    UTILITY_COLS_ORDER,
    REFERENCE_YEAR
)

In [49]:
df

Scenario,Year,Flow type,CO2,Methane,N2O,F-gases,CO2 (fossil) CCU/CCS,CO2 (bio) CCU/CCS,Electricity,Electricity_peak,Natural Gas,Hydrogen ( >98% vol.%) (LHV),Hydrogen ( <98% vol.%) (LHV),Heat,Residual gases,Coal and coal products,Oil and oil products,Biomass (liquid),Biomass (solid),Green gas,Waste (fossil),Waste (bio),Ammonia,Methanol,Other syn fuel and raw materials,Other
str,str,str,i64,null,null,null,null,null,f64,f64,i64,null,null,i64,i64,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""demand""",null,null,null,null,null,null,691.2,185.5,-73,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""captive use""",null,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""production""",1758,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""supply""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2035""","""demand""",null,null,null,null,null,null,1303.2,185.5,-259,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Midden""","""2040""","""supply""",null,null,null,null,null,null,null,null,null,null,null,null,1879,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2050""","""demand""",null,null,null,null,null,null,1303.2,185.5,-259,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2050""","""captive use""",null,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null


In [10]:
prod = pl.read_excel('/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/Shell Chemie Moerdijk.xlsx',
              sheet_name=f'Production')

Could not determine dtype for column 11, falling back to string


In [11]:
prod

Name,Year,Status,Production type,Thermal unit subtype,Nominal power (MW),Operating power (MW),Grid level,Storylines,Description,Primary fuel,Secondary fuel,Minimum load (%),Efficiency (%),Must run,Must run period,Must run power (%),Is CHP,CHP heat efficiency (%)
str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64,bool,str,str,bool,i64
"""Shell Chemie Moerdijk""","""2023""","""Operational""","""Solar photovoltaic energy""",null,27,25,"""25-50 kV""","""Electrification""","""PV Site connected to OP-05 (30…",null,null,null,null,false,null,null,false,null
"""Shell Chemie Moerdijk""","""2023""","""Operational""","""Thermal unit""","""Combined Cycle Gas Turbine (CC…",38,37,"""25-50 kV""","""Electrification""","""CoGen unit (GE frame 6B) coup…","""Natural Gas""",null,80,31,true,"""All year""","""92""",true,90
"""Scenario""","""Year""","""Capacity""","""FLH""","""Efficiency""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""37""","""7000""","""31""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2035""","""37""","""7000""","""31""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2040""","""37""","""7000""","""31""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2050""","""0""","""0""","""31""",null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [12]:
import polars as pl


def read_production_table(
    workbook_path: str,
    sheet_name: str = "Production",
) -> pl.DataFrame:
    # Read entire sheet without assuming a header
    raw = pl.read_excel(
        workbook_path,
        sheet_name=sheet_name,
        has_header=False,
    )

    # Find row containing the table header
    header_row = (
        raw.with_row_index()
        .filter(
            (pl.col("column_1") == "Scenario")
            & (pl.col("column_2") == "Year")
        )
        .select("index")
        .item()
    )

    # Extract header values
    header_values = raw.row(header_row)

    # Keep columns until first null header
    n_cols = next(
        (i for i, v in enumerate(header_values) if v is None),
        len(header_values)
    )

    headers = [str(v) for v in header_values[:n_cols]]

    # Data below header
    data = raw.slice(header_row + 1)

    # Keep only relevant columns
    data = data.select(data.columns[:n_cols])

    # Rename columns
    data.columns = headers

    # Stop at first completely empty row
    empty_mask = pl.all_horizontal(
        [pl.col(c).is_null() for c in data.columns]
    )

    empty_rows = (
        data.with_row_index()
        .filter(empty_mask)
        .select("index")
        .to_series()
        .to_list()
    )

    if empty_rows:
        data = data.slice(0, empty_rows[0])

    return data

In [13]:
production = read_production_table(test_file_path)

In [14]:
production

Scenario,Year,Capacity,FLH,Efficiency
str,str,str,str,str
"""Midden""","""2030""","""37""","""7000""","""31"""
"""Midden""","""2035""","""37""","""7000""","""31"""
"""Midden""","""2040""","""37""","""7000""","""31"""
"""Midden""","""2050""","""0""","""0""","""31"""


In [59]:
def aggregate_scenarios_flow_types(all_df: pl.DataFrame) -> pl.DataFrame:

    value_cols = [c for c in all_df.columns if c not in ('Scenario', 'Year', 'Flow type')]

    production = all_df.filter(pl.col('Flow type') == 'production')

    demand = (
        all_df.filter(pl.col('Flow type').is_in(['demand', 'captive use']))
        .group_by(['Scenario', 'Year'])
        .agg([pl.col(c).sum() for c in value_cols])
        .with_columns(pl.lit('demand').alias('Flow type'))
        .select(all_df.columns)  # restore original column order
    )

    result = pl.concat([production, demand])
    return result


def read_all_scenario_sheets(
    workbook_path: str,
    emission_cols: list[str],
    energy_cols: list[str],
    reference_year: int,
    aggregate_flow_types:bool = True
) -> pl.DataFrame:
    """
    Read all Scenario X sheets and return a single dataframe.
    """

    wb = load_workbook(workbook_path, read_only=True)

    scenario_sheets = [
        sheet
        for sheet in wb.sheetnames
        if sheet.startswith("Scenario ")
    ]

    dfs = [
        read_scenario_sheet(
            workbook_path=workbook_path,
            sheet_name=sheet,
            emission_cols=emission_cols,
            energy_cols=energy_cols,
            reference_year=reference_year,
        )
        for sheet in scenario_sheets
    ]

    if not dfs:
        return pl.DataFrame()
    
    if aggregate_flow_types:
        all_df = pl.concat(dfs, how="vertical_relaxed")
        return aggregate_scenarios_flow_types(all_df)

    return pl.concat(dfs, how="vertical_relaxed")

In [60]:
all_df = read_all_scenario_sheets(test_file_path, 
                                  emission_cols=EMISSION_COLS_ORDER, 
                                  energy_cols=UTILITY_COLS_ORDER, 
                                  reference_year=REFERENCE_YEAR)

In [61]:
all_df

Scenario,Year,Flow type,CO2,Methane,N2O,F-gases,CO2 (fossil) CCU/CCS,CO2 (bio) CCU/CCS,Electricity,Electricity_peak,Natural Gas,Hydrogen ( >98% vol.%) (LHV),Hydrogen ( <98% vol.%) (LHV),Heat,Residual gases,Coal and coal products,Oil and oil products,Biomass (liquid),Biomass (solid),Green gas,Waste (fossil),Waste (bio),Ammonia,Methanol,Other syn fuel and raw materials,Other
str,str,str,i64,null,null,null,null,null,f64,f64,i64,null,null,i64,i64,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""production""",1758,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2035""","""production""",1379,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2040""","""production""",1379,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2050""","""production""",1379,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""VT""","""2030""","""production""",1758,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Midden""","""2040""","""demand""",0,null,null,null,null,null,1602.2,185.5,-259,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Waterstof""","""2035""","""demand""",0,null,null,null,null,null,639.0,85.0,2410,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Waterstof""","""2040""","""demand""",0,null,null,null,null,null,639.0,85.0,2410,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null


In [57]:
value_cols = [c for c in all_df.columns if c not in ('Scenario', 'Year', 'Flow type')]

production = all_df.filter(pl.col('Flow type') == 'production')

demand = (
    all_df.filter(pl.col('Flow type').is_in(['demand', 'captive use']))
      .group_by(['Scenario', 'Year'])
      .agg([pl.col(c).sum() for c in value_cols])
      .with_columns(pl.lit('demand').alias('Flow type'))
      .select(all_df.columns)  # restore original column order
)

result = pl.concat([production, demand])
result

Scenario,Year,Flow type,CO2,Methane,N2O,F-gases,CO2 (fossil) CCU/CCS,CO2 (bio) CCU/CCS,Electricity,Electricity_peak,Natural Gas,Hydrogen ( >98% vol.%) (LHV),Hydrogen ( <98% vol.%) (LHV),Heat,Residual gases,Coal and coal products,Oil and oil products,Biomass (liquid),Biomass (solid),Green gas,Waste (fossil),Waste (bio),Ammonia,Methanol,Other syn fuel and raw materials,Other
str,str,str,i64,null,null,null,null,null,f64,f64,i64,null,null,i64,i64,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2030""","""production""",1758,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2035""","""production""",1379,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2040""","""production""",1379,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2050""","""production""",1379,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
"""VT""","""2030""","""production""",1758,null,null,null,null,null,299.0,null,null,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Groen gas""","""2040""","""demand""",0,null,null,null,null,null,940.0,85.0,2410,null,null,2580,5790,null,null,null,null,null,null,null,null,null,null,null
"""Groen gas""","""2050""","""demand""",0,null,null,null,null,null,940.0,85.0,2410,null,null,2580,5790,null,null,null,null,null,null,null,null,null,null,null
"""Midden""","""2050""","""demand""",0,null,null,null,null,null,1602.2,185.5,-259,null,null,2580,5590,null,null,null,null,null,null,null,null,null,null,null


### read the curves for the cluster + sector

In [18]:
curves = pl.read_excel("/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/ctm format curves.xlsx")

In [19]:
curves

Netbeheerder,Cluster,Sector,Energiedrager,Scenario,2024,2030,2035,2040,2045,2050,Eenheid
str,str,str,str,str,f64,f64,f64,f64,f64,f64,str
"""Stedin""","""Overig""","""Other""","""elektriciteit""","""Hoekpunt E""",0.0,25.25,75.75,126.25,176.75,227.25,"""GWh"""
"""Stedin""","""Overig""","""Aluminium""","""elektriciteit""","""Hoekpunt E""",11.943028,11.956511,12.013414,12.113737,12.220915,12.334948,"""GWh"""
"""Stedin""","""Overig""","""Construction""","""elektriciteit""","""Hoekpunt E""",211.260716,219.680422,235.338593,258.235229,282.03666,306.742886,"""GWh"""
"""Stedin""","""Overig""","""Food""","""elektriciteit""","""Hoekpunt E""",658.803037,941.373607,1408.325146,1880.653685,2352.982224,2830.687763,"""GWh"""
"""Stedin""","""Overig""","""Machinery""","""elektriciteit""","""Hoekpunt E""",180.747607,198.511124,231.458019,278.345854,327.321139,378.896596,"""GWh"""
…,…,…,…,…,…,…,…,…,…,…,…
"""Enexis""","""overig""","""Transport equipment""","""gas""","""Vertraagd""",54.332,25.536,21.733,21.733,21.733,21.733,"""GWh"""
"""Enexis""","""overig""","""Transport equipment""","""waterstof""","""Vertraagd""",0.0,0.0,0.0,0.0,0.0,0.0,"""GWh"""
"""Enexis""","""overig""","""Wood and wood products""","""elektriciteit""","""Vertraagd""",100.751,108.599,109.636,109.636,109.636,109.636,"""GWh"""


In [83]:
SCENARIO_YEARS = ['2030', '2035', '2040', '2050']
REFERENCE_YEAR = '2024'

ALL_YEARS = [REFERENCE_YEAR] + SCENARIO_YEARS

sums = curves.group_by(['Cluster', 'Sector', 'Scenario', 'Energiedrager']
                ).agg(
                    [pl.col(i).sum() for i in ALL_YEARS]
                ).sort(['Cluster', 'Sector', 'Scenario', 'Energiedrager'])


sums = sums.with_columns(replaced=pl.col("Cluster").replace('overig', 'Cluster 6').replace('Overig', 'Cluster 6'))
sums = sums.with_columns(pl.col('replaced').alias('Cluster')).drop('replaced')



In [84]:
sums

Cluster,Sector,Scenario,Energiedrager,2024,2030,2035,2040,2050
str,str,str,str,f64,f64,f64,f64,f64
"""Chemelot""","""Food""","""Hoekpunt_E""","""elektriciteit""",0.688,0.801,0.816,0.816,0.816
"""Chemelot""","""Food""","""Hoekpunt_E""","""gas""",0.321,0.038,0.0,0.0,0.0
"""Chemelot""","""Food""","""Hoekpunt_E""","""waterstof""",0.0,0.0,0.0,0.0,0.0
"""Chemelot""","""Food""","""Hoekpunt_G""","""elektriciteit""",0.688,0.801,0.816,0.816,0.816
"""Chemelot""","""Food""","""Hoekpunt_G""","""gas""",0.321,0.038,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…
"""Cluster 6""","""Wood and wood products""","""Midden""","""gas""",59.232,17.177,11.846,11.254,5.923
"""Cluster 6""","""Wood and wood products""","""Midden""","""waterstof""",0.0,0.0,0.0,0.592,5.923
"""Cluster 6""","""Wood and wood products""","""Vertraagd""","""elektriciteit""",100.751,108.599,109.636,109.636,109.636


### the mapping

In [22]:
maps = pl.read_excel('/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/CTM-DSH site mapping.xlsx')

Could not determine dtype for column 4, falling back to string


In [28]:
MARKER_VALUES = ['Bestaande niet-bottom-up sites', 'Bottom-up sites', 'New sites']  

df = maps.with_columns(
    pl.when(pl.col('Name').is_in(MARKER_VALUES))
      .then(pl.col('Name'))
      .otherwise(None)
      .forward_fill()
      .alias('category')
)

df = df.filter(~pl.col('Name').is_in(MARKER_VALUES))

In [29]:
df.write_csv('/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/mapping.csv')

In [65]:
from pathlib import Path

def read_and_transform_mapping(
        excel_path: str,
        markers: list = ['Bestaande niet-bottom-up sites', 'Bottom-up sites', 'New sites'],
        marker_column_name:str = 'Name',
        save_file: bool = False,
        save_path: str = '' # if empty defaults to location of original file
) -> pl.DataFrame:
    
    maps = pl.read_excel(excel_path)
    df = maps.with_columns(
        pl.when(pl.col(marker_column_name).is_in(markers))
        .then(pl.col(marker_column_name))
        .otherwise(None)
        .forward_fill()
        .alias('category')
    )

    df = df.filter(~pl.col(marker_column_name).is_in(markers))

    result = df.with_columns([pl.when(pl.col('category')=='New sites')
                 .then(pl.lit(True))
                 .otherwise(False)
                 .alias('New site'),
                 
                 pl.when(pl.col('category')=='Bottom-up sites')
                 .then(pl.lit(True))
                 .otherwise(False)
                 .alias('Bottom-up')
                 ])

    if save_file:
        if save_path == '':
            save_path = Path(excel_path).parent
            
        result.write_csv(f'{save_path}/mapping.csv')

    return result

In [66]:
res = read_and_transform_mapping(excel_path='/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/CTM-DSH site mapping.xlsx')

Could not determine dtype for column 4, falling back to string


In [69]:
res

Name,Name reformatted,Sector,Cluster,API input name,DSH plant name,DSH plant id,category,New site,Bottom-up
str,str,str,str,str,str,str,str,bool,bool
"""A12 CPP Petrogas EP Netherland…","""a12_cpp_petrogas_ep_netherland…","""Other""","""Cluster 6""","""other&&cluster_6&&a12_cpp_petr…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Aardgasbuffer Zuidwending""","""aardgasbuffer_zuidwending""","""Other""","""Cluster 6""","""other&&cluster_6&&aardgasbuffe…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Abbott Healthcare Products""","""abbott_healthcare_products""","""Other chemicals""","""Cluster 6""","""other_chemicals&&cluster_6&&ab…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Abbott Laboratories""","""abbott_laboratories""","""Food""","""Cluster 6""","""food&&cluster_6&&abbott_labora…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""ADM Europoort""","""adm_europoort""","""Food""","""Rotterdam-Moerdijk""","""food&&rotterdam-moerdijk&&adm_…","""ADM Europoort""","""527be9a2-5258-4cda-89cb-b48520…","""Bestaande niet-bottom-up sites""",false,false
…,…,…,…,…,…,…,…,…,…
"""##new_cc_site230##""","""##new_cc_site230##""","""Gebruiker stuurt op via API""","""Gebruiker stuurt op via API""","""##new_cc_site230##""",null,null,"""New sites""",true,false
"""##new_cc_site231##""","""##new_cc_site231##""","""Gebruiker stuurt op via API""","""Gebruiker stuurt op via API""","""##new_cc_site231##""",null,null,"""New sites""",true,false
"""##new_cc_site232##""","""##new_cc_site232##""","""Gebruiker stuurt op via API""","""Gebruiker stuurt op via API""","""##new_cc_site232##""",null,null,"""New sites""",true,false


In [ ]:
# res2 = res.with_columns([pl.when(pl.col('category')=='New sites')
#                  .then(pl.lit(True))
#                  .otherwise(False)
#                  .alias('New site'),
                 
#                  pl.when(pl.col('category')=='Bottom-up sites')
#                  .then(pl.lit(True))
#                  .otherwise(False)
#                  .alias('Bottom-up')
#                  ])

In [44]:
res2

Name,Name reformatted,Sector,Cluster,API input name,DSH plant name,DSH plant id,category,New site,Bottom-up
str,str,str,str,str,str,str,str,bool,bool
"""A12 CPP Petrogas EP Netherland…","""a12_cpp_petrogas_ep_netherland…","""Other""","""Cluster 6""","""other&&cluster_6&&a12_cpp_petr…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Aardgasbuffer Zuidwending""","""aardgasbuffer_zuidwending""","""Other""","""Cluster 6""","""other&&cluster_6&&aardgasbuffe…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Abbott Healthcare Products""","""abbott_healthcare_products""","""Other chemicals""","""Cluster 6""","""other_chemicals&&cluster_6&&ab…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Abbott Laboratories""","""abbott_laboratories""","""Food""","""Cluster 6""","""food&&cluster_6&&abbott_labora…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""ADM Europoort""","""adm_europoort""","""Food""","""Rotterdam-Moerdijk""","""food&&rotterdam-moerdijk&&adm_…","""ADM Europoort""","""527be9a2-5258-4cda-89cb-b48520…","""Bestaande niet-bottom-up sites""",false,false
…,…,…,…,…,…,…,…,…,…
"""##new_cc_site230##""","""##new_cc_site230##""","""Gebruiker stuurt op via API""","""Gebruiker stuurt op via API""","""##new_cc_site230##""",null,null,"""New sites""",true,false
"""##new_cc_site231##""","""##new_cc_site231##""","""Gebruiker stuurt op via API""","""Gebruiker stuurt op via API""","""##new_cc_site231##""",null,null,"""New sites""",true,false
"""##new_cc_site232##""","""##new_cc_site232##""","""Gebruiker stuurt op via API""","""Gebruiker stuurt op via API""","""##new_cc_site232##""",null,null,"""New sites""",true,false
